In [ ]:
'''
This is Auto ML project , in which user can upload their dataset and then all ML spteps will be done automatically.
For this User just need to upload the dataset and select the target variable.

Here We code and test all the functions that are used  for our project.
This file is part of the project:
1. Load the dataset 
2. Analyze the dataset find the traget variable
3. Preprocess the data
4. Train the different models for the data
5. Evaluate the models and select the best one
6. Preform hyperparameter tuning on the best model
7. Save the model for future use
8. Create a web interface for the user to interact with the model
'''

Preprocessing

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings(action="ignore")

class Preprocess:
    def __init__(self, df, target):
        self.df = df
        self.target = target
        if self.df[target].dtype == 'object':
            self.type = 'categorical'
        else:
            self.type = 'numerical'
    def handle_duplicate_rows(self):
        # Check for duplicate rows
        if self.df.duplicated().any():
            # Remove duplicate rows
            self.df = self.df.drop_duplicates()
        
        return self.df
    def handle_missing_values(self):
        for column in self.df.columns:
            if self.df[column].isnull().any():
                if self.df[column].dtype == 'object':
                    # Fill categorical missing values with the mode
                    self.df[column].fillna(self.df[column].mode()[0], inplace=True)
                else:
                    # Fill numerical missing values with the mean
                    self.df[column].fillna(self.df[column].median(), inplace=True)

        return self.df
    def handle_outliers(self):
        for column in self.df.select_dtypes(include=['float64', 'int64']).columns:
            # Calculate the IQR
            Q1 = self.df[column].quantile(0.25)
            Q3 = self.df[column].quantile(0.75)
            IQR = Q3 - Q1
            
            # Define bounds for outliers
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Replace outliers with the median
            self.df[column] = self.df[column].apply(lambda x: x if (x >= lower_bound and x <= upper_bound) else self.df[column].median())
        
        return self.df
    # def handel_categorical_data(self):
    #     # Convert categorical variables to numerical
    #     self.df = pd.get_dummies(self.df, drop_first=True)
        
    #     # Separate features and target variable
    #     X = self.df.drop(columns=[self.target])
    #     y = self.df[self.target]
        
    #     return X, y
    def handle_categorical_data(self):
        # Convert categorical variables to numerical
        for column in self.df.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            self.df[column] = le.fit_transform(self.df[column])
        
        # Separate features and target variable
        X = self.df.drop(columns=[self.target])
        y = self.df[self.target]
        
        return X, y, le
    
    def scale_data(self, X):
        # Scale the data using StandardScaler
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        self.X = pd.DataFrame(X_scaled, columns=X.columns)
        
        return X, scaler
    def preprocess(self):
        # Handle duplicate rows
        self.df = self.handle_duplicate_rows()
        # Handle missing values
        self.df = self.handle_missing_values()
        
        # Handle outliers
        self.df = self.handle_outliers()
        
        # Handle categorical data
        X, y, le = self.handle_categorical_data()
        
        # Scale the data
        X_scaled, scaler = self.scale_data(X)
        
        return X_scaled, y, le, scaler ,self.type

In [2]:
df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')
preprocessor = Preprocess(df, target='Type')



In [3]:
X_scaled, y, le, scaler ,type= preprocessor.preprocess()

In [4]:
X_scaled.head(), y.head(), le.classes_, scaler.mean_

(   Shape  Cut  Color  Clarity  Carat Weight  Length/Width Ratio  Depth %  \
 0      1    2      2        5          1.03                1.02     65.8   
 1      6    2      1        2          1.20                1.65     62.5   
 2      5    2      1        2          1.19                1.41     63.1   
 3      3    2      0        1          1.00                1.18     61.7   
 4      8    2      4        2          1.01                1.35     69.4   
 
    Table %  Polish  Symmetry  Girdle  Culet  Length  Width  Height   Price  \
 0     59.0       0         2       7      1    7.09   6.95    4.57  2640.0   
 1     58.0       2         2       7      1    9.64   5.86    3.66  1070.0   
 2     63.0       2         2       7      1    8.44   6.00    3.79  1070.0   
 3     58.0       0         0      13      1    5.85   6.89    4.25  7110.0   
 4     66.0       0         2      14      1    6.80   5.05    3.50  3050.0   
 
    Fluorescence  
 0             0  
 1             0  
 2 

In [5]:
le.classes_

array(['Faint', 'Medium', 'Strong'], dtype=object)

In [4]:
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.svm import SVC,SVR

In [18]:
# Collection of models
'''
We Have add functionality to identify the type of target variable and then train the models accordingly.
'''


Regression_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Support Vector Regressor': SVR()
}
Classification_models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree Classifier': DecisionTreeClassifier(),
    'Random Forest Classifier': RandomForestClassifier(),
    'Support Vector Classifier': SVC()
}
def train_models(X, y,type):
    results = {}
    models=[]
    if type == 'categorical':
        # Classification task

        for name, model in Classification_models.items():
            model.fit(X, y)
            score = model.score(X, y)
            results[name] = score
            models.append(model)
        return results , models
    else:
        # Regression task
       
        for name, model in Regression_models.items():
            model.fit(X, y)
            score = model.score(X, y)
            results[name] = score
            models.append(model)
        return results, models
   

In [19]:
trained_models,models = train_models(X_scaled, y,type )


In [20]:
def best_model(results,models):
    # Find the best model based on the highest score
    best_model_name = max(results, key=results.get)
    best_model_score = results[best_model_name]
    best_model = models[list(results.keys()).index(best_model_name)]
    
    return best_model_name, best_model_score , best_model

In [21]:
best_model_name, best_model_score, best_model = best_model(trained_models,models)
print(f"Best Model: {best_model_name} with score: {best_model_score}")

Best Model: Decision Tree Classifier with score: 1.0


In [23]:
def hyperparameter_tuning(model, X, y):
    from sklearn.model_selection import GridSearchCV
    
    # Define hyperparameters to tune
    if isinstance(model, RandomForestRegressor) or isinstance(model, RandomForestClassifier):
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    elif isinstance(model, DecisionTreeRegressor) or isinstance(model, DecisionTreeClassifier):
        param_grid = {
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    else:
        param_grid = {}
    
    grid_search = GridSearchCV(model, param_grid, cv=5)
    grid_search.fit(X, y)
    
    return grid_search.best_estimator_

In [24]:
model= hyperparameter_tuning(best_model, X_scaled, y)

In [32]:
print(model.predict(X_scaled[0:5]))
print(y[0:5])

[1 1 1 0 0]
0    1
1    1
2    1
3    0
4    0
Name: Type, dtype: int64


In [ ]:
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.svm import SVC,SVR
class ModelsTrainer:
    
    def __init__(self,X_scaled, y, le, scaler ,type):
        self.X_scaled = X_scaled
        self.y = y
        self.le = le
        self.scaler = scaler
        self.type = type
        
    Regression_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Support Vector Regressor': SVR()
    }
    Classification_models = {
        'Logistic Regression': LogisticRegression(),
        'Decision Tree Classifier': DecisionTreeClassifier(),
        'Random Forest Classifier': RandomForestClassifier(),
        'Support Vector Classifier': SVC()
    }
    def train_models(self):
        results = {}
        models = []
        if self.type == 'categorical':
            # Classification task
            for name, model in self.Classification_models.items():
                model.fit(self.X_scaled, self.y)
                score = model.score(self.X_scaled, self.y)
                results[name] = score
                models.append(model)
            return results, models
        else:
            # Regression task
            for name, model in self.Regression_models.items():
                model.fit(self.X_scaled, self.y)
                score = model.score(self.X_scaled, self.y)
                results[name] = score
                models.append(model)
            return results, models
        
    def best_model(self,results,models):
        # Find the best model based on the highest score
        best_model_name = max(results, key=results.get)
        best_model_score = results[best_model_name]
        best_model = models[list(results.keys()).index(best_model_name)]
        
        return best_model_name, best_model_score , best_model
    def hyperparameter_tuning(self, model):
        from sklearn.model_selection import GridSearchCV
        
        # Define hyperparameters to tune
        if isinstance(model, RandomForestRegressor) or isinstance(model, RandomForestClassifier):
            param_grid = {
                'n_estimators': [50, 100, 200],
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5, 10]
            }
        elif isinstance(model, DecisionTreeRegressor) or isinstance(model, DecisionTreeClassifier):
            param_grid = {
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5, 10]
            }
        else:
            param_grid = {}
        
        grid_search = GridSearchCV(model, param_grid, cv=5)
        grid_search.fit(self.X_scaled, self.y)
        
        return grid_search.best_estimator_
    
    def train(self):
        trained_models, models = self.train_models()
        best_model_name, best_model_score, best_model = self.best_model(trained_models, models)
        print(f"Best Model: {best_model_name} with score: {best_model_score}")
        tuned_model = self.hyperparameter_tuning(best_model)
        
        return tuned_model

In [6]:
from Preprocess import Preprocess

In [7]:
import pandas as pd

In [8]:
df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')

In [9]:
pre= Preprocess(df, target='Shape')

Preprocessing Started ...


In [10]:
X_scaled,y,le,scaler,type= pre.preprocess()

Preprocessing Completed


In [73]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [86]:
x_train,x_test,y_train,y_test=train_test_split(X_scaled,y,random_state=42)

In [68]:
model=SVC()

In [76]:
pred=model.predict(x_test)

In [77]:
from sklearn.metrics import classification_report
print(classification_report(y_true=y_test,y_pred=pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1499
           1       0.00      0.00      0.00        76
           2       0.00      0.00      0.00        46

    accuracy                           0.92      1621
   macro avg       0.31      0.33      0.32      1621
weighted avg       0.86      0.92      0.89      1621



In [75]:
model.fit(x_train,y_train)

SVC()

In [7]:
X_scaled

,Shape,Cut,Color,Clarity,Carat Weight,Length/Width Ratio,Depth %,Table %,Polish,Symmetry,Girdle,Culet,Length,Width,Height,Price,Fluorescence
0,-1.903174,0.152793,0.265078,1.883202,-0.182620,-0.953751,0.333135,-0.592977,-0.294601,1.600608,-0.633124,-0.060796,-0.321532,1.189991,1.764889,-0.260506,-0.279248
1,0.169261,0.152793,-0.434529,-0.723628,2.648104,1.001958,-0.361355,-0.809718,3.457074,1.600608,-0.633124,-0.060796,1.317836,-0.089089,-0.300595,-1.155956,-0.279248
2,-0.245226,0.152793,-0.434529,-0.723628,2.481591,0.256926,-0.235084,0.273985,3.457074,1.600608,-0.633124,-0.060796,0.546369,0.075196,-0.005526,-1.155956,-0.279248
3,-1.074200,0.152793,-1.134136,-1.592571,-0.682160,-0.457063,-0.529716,-0.809718,-0.294601,-0.639759,0.547008,-0.060796,-1.118715,1.119583,1.038565,2.288962,-0.279248
4,0.998234,0.152793,1.664292,-0.723628,-0.515646,0.070668,1.090759,0.924207,-0.294601,1.600608,0.743697,-0.060796,-0.507970,-1.039598,-0.663757,-0.026662,-0.279248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6477,0.169261,0.152793,0.265078,0.145315,-0.349133,0.660485,-0.277174,0.273985,-0.294601,1.600608,1.137074,-0.060796,0.649231,-0.405926,-0.641059,0.378287,2.163515
6478,-1.074200,0.152793,-0.434529,1.883202,-0.682160,-0.519149,-1.245251,-0.592977,-0.294601,-0.639759,0.547008,-0.060796,-0.990137,1.236930,0.630008,0.920120,4.606278
6479,0.583747,0.152793,0.964685,1.014259,-0.182620,-0.984794,1.764204,2.007911,-0.294601,-0.639759,-0.829812,-0.060796,-1.285866,-0.499803,0.471124,0.098815,-0.279248
6480,0.998234,0.152793,-0.434529,-0.723628,0.316920,0.319012,0.754037,0.057245,-0.294601,-0.639759,0.350319,-0.060796,-0.225099,-1.039598,-0.845337,-1.167363,-0.279248


In [21]:
new=df.head()

In [ ]:
new

,Shape,Cut,Color,Clarity,Carat Weight,Length/Width Ratio,Depth %,Table %,Polish,Symmetry,Girdle,Culet,Length,Width,Height,Price,Type,Fluorescence
0,Cushion Modified,Ideal,F,VVS2,1.84,1.02,65.8,59.0,Excellent,Very Good,Medium to Thick,NaN,7.09,6.95,4.57,2640,GIA Lab-Grown,NaN
1,Pear,NaN,E,VS1,1.20,1.65,62.5,58.0,Very Good,Very Good,Medium to Thick,NaN,9.64,5.86,3.66,1070,GIA Lab-Grown,NaN
2,Oval,NaN,E,VS1,1.19,1.41,63.1,63.0,Very Good,Very Good,Medium to Thick,NaN,8.44,6.00,3.79,1070,GIA Lab-Grown,NaN
3,Heart,NaN,D,IF,1.00,1.18,61.7,58.0,Excellent,Excellent,Slightly Thick to Very Thick,NaN,5.85,6.89,4.25,7110,GIA,Faint
4,Radiant,NaN,H,VS1,1.01,1.35,69.4,66.0,Excellent,Very Good,Thick,NaN,6.80,5.05,3.50,3050,GIA,NaN


In [25]:
from ModelsTrainer import ModelsTrainer
from Preprocess import Preprocess
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings(action="ignore")

df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')
preprocessor = Preprocess(df, target='Price')
X_scaled, y, le, scaler ,type= preprocessor.preprocess()
trainer = ModelsTrainer(X_scaled, y, le, scaler, type)
res,tuned_model = trainer.train()

print(scaler)

Preprocessing Started ...
Handling Numerical Outliers ...
Handling Categrical Outliers ...
scaler_dict created 
Preprocessing Completed


2025-09-04 23:30:26.562 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-04 23:30:26.659 
  command:

    streamlit run /Users/abhishekkanade/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-09-04 23:30:26.660 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Best Model: Random Forest Regressor with score: 0.8632798745941684
Tuned Model: RandomForestRegressor(max_depth=10, n_jobs=1, random_state=42)
{'Shape': StandardScaler(), 'Cut': StandardScaler(), 'Color': StandardScaler(), 'Clarity': StandardScaler(), 'Carat Weight': StandardScaler(), 'Length/Width Ratio': StandardScaler(), 'Depth %': StandardScaler(), 'Table %': StandardScaler(), 'Polish': StandardScaler(), 'Symmetry': StandardScaler(), 'Girdle': StandardScaler(), 'Culet': StandardScaler(), 'Length': StandardScaler(), 'Width': StandardScaler(), 'Height': StandardScaler(), 'Type': StandardScaler(), 'Fluorescence': StandardScaler()}


In [38]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
def scale_data(X):
        scaler_dict = {}
        X_scaled = X.copy()
        for col in X.columns:
            scaler = StandardScaler()
            X_scaled[[col]] = scaler.fit_transform(X[[col]])
            scaler_dict[col] = scaler
        print('scaler_dict created ')    
        return X_scaled, scaler_dict

def handle_categorical_data(df,target):
        le_dict = {}
        for column in df.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            df[column] = le.fit_transform(df[column])
            le_dict[column] = le
        X = df.drop(columns=[target])
        y = df[target]
        return X, y, le_dict

In [39]:
X,_,_=handle_categorical_data(df,'Shape')

In [41]:
X,scaler=scale_data(X)

scaler_dict created 


In [42]:
X

,Cut,Color,Clarity,Carat Weight,Length/Width Ratio,Depth %,Table %,Polish,Symmetry,Girdle,Culet,Length,Width,Height,Price,Type,Fluorescence
0,-1.171469,0.262796,1.875142,1.192695,-0.950462,0.333870,-0.594252,-0.299119,1.573357,-0.634418,0.638934,-0.342540,0.739904,1.141678,-0.271141,0.262540,0.453652
1,0.672010,-0.435373,-0.724323,-0.070610,0.983943,-0.357545,-0.804344,3.299747,1.573357,-0.634418,0.638934,1.180271,-0.207046,-0.376572,-0.749772,0.262540,0.453652
2,0.672010,-0.435373,-0.724323,-0.090349,0.247027,-0.231833,0.246114,3.299747,1.573357,-0.634418,0.638934,0.463654,-0.085420,-0.159679,-0.749772,0.262540,0.453652
3,0.672010,-1.133543,-1.590811,-0.465393,-0.459184,-0.525161,-0.804344,-0.299119,-0.643289,0.520716,0.638934,-1.083043,0.687779,0.607788,1.091587,-0.888623,-2.566815
4,0.672010,1.659135,-0.724323,-0.445654,0.062798,1.088140,0.876388,-0.299119,1.573357,0.713238,0.638934,-0.515722,-0.910744,-0.643517,-0.146148,-0.888623,0.453652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6480,0.672010,0.262796,0.142166,-0.425914,0.646190,-0.273737,0.246114,-0.299119,1.573357,1.098283,0.638934,0.559203,-0.441612,-0.626833,0.070304,-0.888623,-1.559993
6481,0.672010,-0.435373,1.875142,-0.465393,-0.520594,-1.237527,-0.594252,-0.299119,-0.643289,0.520716,0.638934,-0.963607,0.774655,0.307475,0.359921,-0.888623,-0.553170
6482,-1.171469,0.960965,1.008654,-0.406175,-0.981166,1.758603,1.926846,-0.299119,-0.643289,-0.826941,0.638934,-1.238310,-0.511113,0.190686,-0.079078,-0.888623,0.453652
6483,0.672010,-0.435373,-0.724323,-0.346958,0.308437,0.752909,0.036022,-0.299119,-0.643289,0.328194,0.638934,-0.252963,-0.910744,-0.776990,-0.755869,0.262540,0.453652


array([-0.24972957])

In [3]:
plt.figure(figsize=(10, 6))
plt.barh(list(res.keys()), list(res.values()), color='skyblue')
plt.xlabel('Model Score')
plt.title('Model Performance Comparison')
plt.show()

In [4]:
tuned_model

RandomForestClassifier(n_jobs=1, random_state=42)

In [26]:
import pickle
# Save the model to a file
with open('tuned_model.pkl', 'wb') as file:
    pickle.dump(tuned_model, file)  
with open('label.pkl','wb') as file:
    pickle.dump(le, file)    
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)  

In [11]:
import shap

# Choose explainer based on model type
if hasattr(tuned_model, "feature_importances_"):
    explainer = shap.TreeExplainer(tuned_model)
    shap_values = explainer.shap_values(X_scaled)
else:
    explainer = shap.Explainer(tuned_model.predict, X_scaled)
    shap_values = explainer(X_scaled)

# Visualize SHAP values
shap.summary_plot(shap_values, X_scaled)


KeyboardInterrupt: 

In [13]:
n,m=le['Shape'].inverse_transform(tuned_model.predict(x_test)),le['Shape'].inverse_transform(y_test)

NameError: name 'x_test' is not defined

In [5]:
import pandas as pd
import pickle
import os
from ScriptGenerator import ScriptGenerator

# Load the dataset
df=pd.read_csv('/Users/abhishekkanade/Documents/Data Science/diamonds dataset.csv')



sg=ScriptGenerator(df)
info=sg.get_info()
target='Price'
# Path 
path=os.getcwd()

# model_path=path+"/tuned_model.pkl"
# lable_endocer=path+'/label.pkl'
# scaler = path+'/scaler.pkl'
# # Load model,encoder,scalar
# model = pickle.load(open(model_path, 'rb'))
# encoder = pickle.load(open(lable_endocer, 'rb'))
# scaler = pickle.load(open(scaler, 'rb'))



# data=[]
# for col in info.keys():
#     if col == target:
#         continue
#     if info[col] == 'object':
#         while True:
#             x=input(f"Enter {col} value from {encoder[col].classes_}: ")
#             try:
#                 x=encoder[col].transform([x])[0]
#                 data.append(scaler[col].transform([[x]])[0][0])
#                 break
#             except:
#                 print(f"Invalid value for {col}. Please try again.")
                 
#     if info[col] == 'int64' or info[col] == 'float64':
#         while True:
#             try:
#                 x=float(input(f"Enter {col} value: "))
#                 data.append(scaler[col].transform([[x]])[0][0])  
#                 break
#             except:
#                 print(f"Invalid value for {col}. Please try again.")

# if info[target] == 'object':
#     print(f"Predicted {target} is :{encoder[target].inverse_transform([model.predict([data])])[0]}")  
# else:
#     print(f"Predicted {target} is : {model.predict([data])[0]}")        


In [27]:
import os

path=os.getcwd()
scaler = path+'/scaler.pkl'
scaler = pickle.load(open(scaler, 'rb'))

scaler

{'Shape': StandardScaler(),
 'Cut': StandardScaler(),
 'Color': StandardScaler(),
 'Clarity': StandardScaler(),
 'Carat Weight': StandardScaler(),
 'Length/Width Ratio': StandardScaler(),
 'Depth %': StandardScaler(),
 'Table %': StandardScaler(),
 'Polish': StandardScaler(),
 'Symmetry': StandardScaler(),
 'Girdle': StandardScaler(),
 'Culet': StandardScaler(),
 'Length': StandardScaler(),
 'Width': StandardScaler(),
 'Height': StandardScaler(),
 'Type': StandardScaler(),
 'Fluorescence': StandardScaler()}

In [70]:
encoder['Cut'].inverse_transform([model.predict([data])])[0]

'Ideal'

In [53]:
from ModelsTrainer import ModelsTrainer
from Preprocess import Preprocess
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings(action="ignore")

df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')
preprocessor = Preprocess(df, target='Cut')
X_scaled, y, le, scaler ,type= preprocessor.preprocess()
trainer = ModelsTrainer(X_scaled, y, le, scaler, type)
res,tuned_model = trainer.train()

print(scaler)

Preprocessing Started ...
Handling Numerical Outliers ...
Handling Categrical Outliers ...
scaler_dict created 
Preprocessing Completed


2025-08-30 16:23:59.816 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-30 16:23:59.816 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Best Model: Support Vector Classifier with score: 0.9992921743779377
Tuned Model: SVC(probability=True)
{'Shape': StandardScaler(), 'Color': StandardScaler(), 'Clarity': StandardScaler(), 'Carat Weight': StandardScaler(), 'Length/Width Ratio': StandardScaler(), 'Depth %': StandardScaler(), 'Table %': StandardScaler(), 'Polish': StandardScaler(), 'Symmetry': StandardScaler(), 'Girdle': StandardScaler(), 'Culet': StandardScaler(), 'Length': StandardScaler(), 'Width': StandardScaler(), 'Height': StandardScaler(), 'Price': StandardScaler(), 'Type': StandardScaler(), 'Fluorescence': StandardScaler()}


In [66]:
scaler['Shape'].transform([[3]])[0][0]

-0.358973598618477

In [2]:
target="Cut"
import pandas as pd
from ScriptGenerator import ScriptGenerator
df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')
sg=ScriptGenerator(df,target)
sg.create_script()


In [3]:
import pandas as pd
import numpy as np


In [4]:

df=pd.read_csv('/Users/abhishekkanade/Documents/notebook/DA/diamonds dataset.csv')
df.isna().mean()

Shape                 0.000771
Cut                   0.663531
Color                 0.000771
Clarity               0.000771
Carat Weight          0.000771
Length/Width Ratio    0.000771
Depth %               0.001079
Table %               0.002621
Polish                0.003084
Symmetry              0.003084
Girdle                0.003392
Culet                 0.708404
Length                0.003084
Width                 0.003084
Height                0.003084
Price                 0.000000
Type                  0.000000
Fluorescence          0.811719
dtype: float64

In [63]:
def remove_low_variance_features(df, threshold=0.1):
    """
    Remove features with variance below the specified threshold.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    threshold (float): The variance threshold below which features will be removed.
    
    Returns:
    pd.DataFrame: DataFrame with low variance features removed.
    """
    df_ = df.select_dtypes(include=[np.number])
    low_variance_cols = [col for col in df_.columns if df_[col].var() < threshold]
    print(f"Removing low variance features: {low_variance_cols}")
    return df.drop(columns=low_variance_cols)

In [64]:
df = remove_low_variance_features(df,0.01)

Removing low variance features: []


In [75]:
def remove_highly_correlated_features(df, threshold=0.1):
    """
    Remove features that are highly correlated with each other.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    threshold (float): The correlation threshold above which one of the correlated features will be removed.
    
    Returns:
    pd.DataFrame: DataFrame with highly correlated features removed.
    """
    df_ = df.select_dtypes(include=[np.number])
    corr_matrix = df_.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
    print(f"Removing highly correlated features: {to_drop}")
    return df.drop(columns=to_drop)

In [76]:
df=remove_highly_correlated_features(df)

Removing highly correlated features: ['Length/Width Ratio', 'Depth %', 'Table %', 'Length', 'Width', 'Price']


In [16]:
mispre=df.isnull().mean()
mispre

Shape                 0.000771
Cut                   0.663531
Color                 0.000771
Clarity               0.000771
Carat Weight          0.000771
Length/Width Ratio    0.000771
Depth %               0.001079
Table %               0.002621
Polish                0.003084
Symmetry              0.003084
Girdle                0.003392
Culet                 0.708404
Length                0.003084
Width                 0.003084
Height                0.003084
Price                 0.000000
Type                  0.000000
Fluorescence          0.811719
dtype: float64

In [20]:
col=mispre[mispre > 0.2].index
df.drop(columns=col, inplace=True)


In [5]:
df['Shape'].unique()
df=df[df['Shape']=='Round']

In [10]:
single_value_cols = [col for col in df.columns if df[col].nunique() <= 1]

In [8]:
def remove_single_value_columns(df):

        """
    Remove columns with a single unique value.
        """
        single_value_cols = [col for col in df.columns if df[col].nunique() <= 1]
        df.drop(columns=single_value_cols, inplace=True)
        if single_value_cols:
            print(f"Removed single-value columns: {single_value_cols}")
           
        return df

In [9]:
df=remove_single_value_columns(df)


In [10]:
df.head()

,Cut,Color,Clarity,Carat Weight,Length/Width Ratio,Depth %,Table %,Polish,Symmetry,Girdle,Culet,Length,Width,Height,Price,Type,Fluorescence
15,Ideal,E,VS1,1.59,1.00,60.9,59.0,Excellent,Excellent,Medium,Pointed,7.52,7.55,4.59,1850,IGI Lab-Grown,NaN
28,Very Good,G,VS2,1.00,1.00,60.3,62.0,Excellent,Excellent,Slightly Thick to Thick,NaN,6.44,6.46,3.89,4020,GIA,Faint
32,Excellent,E,VS1,1.60,1.00,62.0,57.0,Very Good,Excellent,Medium to Slightly Thick,NaN,7.48,7.49,4.64,1800,GIA Lab-Grown,NaN
37,Excellent,D,VS1,1.51,1.00,60.3,58.0,Excellent,Excellent,Thin to Medium,NaN,7.43,7.46,4.49,1770,GIA Lab-Grown,NaN
49,Very Good,E,VS2,1.01,1.01,64.4,57.0,Excellent,Very Good,Slightly Thick to Thick,NaN,6.20,6.27,4.01,4100,GIA,NaN
